In [2]:
import pandas as pd
import numpy as np

def generate_mushroom_dataset(n_samples=5000):
    """
    Generates a synthetic dataset for Porcini and Chanterelle growth probability
    based on environmental sensor data and historical triggers.
    """
    np.random.seed(42)
    
    # 1. Generate Input Features (X)
    data = {
        'air_temp_day': np.random.uniform(5, 38, n_samples),
        'air_temp_night': np.random.uniform(2, 25, n_samples),
        'soil_temp': np.random.uniform(5, 28, n_samples),
        'air_humidity': np.random.uniform(20, 100, n_samples),
        'soil_moisture': np.random.uniform(10, 95, n_samples),
        'wind_speed': np.random.uniform(0, 10, n_samples),
        'rain_cumulative_7d': np.random.uniform(0, 80, n_samples),
        'rain_days_in_window': np.random.randint(0, 8, n_samples),
        'rain_intensity_high': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
        'consecutive_moist_days': np.random.randint(0, 15, n_samples),
        'hours_above_28c': np.random.uniform(0, 24, n_samples),
        'cumulative_thermal_units_7d': np.random.uniform(0, 150, n_samples),
        'evaporation_risk_index': np.random.uniform(0, 1, n_samples),
        'is_peak_season': np.random.choice([0, 1], size=n_samples, p=[0.4, 0.6])
    }

    df = pd.DataFrame(data)

    def calculate_score(row, species='porcini'):
        score = 1.0
        
        # Shared constraints logic
        # Temperature Penalty (Optimal 15-22, Min 10, Max 28)
        t_mid = 18.5
        if row['air_temp_day'] < 10:
            score *= np.exp(-0.5 * ((row['air_temp_day'] - 10)/2)**2)
        elif row['air_temp_day'] > 22:
            score *= np.exp(-0.5 * ((row['air_temp_day'] - 22)/4)**2)
            
        # Rain Trigger (15-30mm is the sweet spot)
        if 15 <= row['rain_cumulative_7d'] <= 30:
            score *= 1.1 # Bonus for trigger
        else:
            score *= np.exp(-0.5 * ((row['rain_cumulative_7d'] - 22.5)/15)**2)
            
        # Moisture Duration (5-10 days)
        if 5 <= row['consecutive_moist_days'] <= 10:
            score *= 1.2
        else:
            score *= 0.5
            
        # Humidity (75-95%)
        if row['air_humidity'] < 75:
            score *= (row['air_humidity'] / 75)
            
        # Wind (1-3 m/s)
        if row['wind_speed'] > 3:
            score *= np.exp(-0.1 * (row['wind_speed'] - 3))

        # Species Specific Kill-Switches
        if species == 'porcini':
            if row['air_temp_day'] >= 32: score = 0 # Fruiting stops
            if row['rain_intensity_high'] == 1: score *= 0.7 # Soil saturation
        
        if species == 'chanterelle':
            if row['air_temp_day'] >= 28: score *= 0.3 # Growth suppressed
            
        # Seasonal Lock
        if row['is_peak_season'] == 0:
            score *= 0.1
            
        return min(max(score, 0), 1.0) # Keep between 0 and 1

    # Apply scoring
    df['porcini_score'] = df.apply(lambda r: calculate_score(r, 'porcini'), axis=1)
    df['chanterelle_score'] = df.apply(lambda r: calculate_score(r, 'chanterelle'), axis=1)

    return df

# Usage
dataset = generate_mushroom_dataset(n_samples=1000)


In [9]:
dataset[dataset["porcini_score"] > 0.5]

,air_temp_day,air_temp_night,soil_temp,air_humidity,soil_moisture,wind_speed,rain_cumulative_7d,rain_days_in_window,rain_intensity_high,consecutive_moist_days,hours_above_28c,cumulative_thermal_units_7d,evaporation_risk_index,is_peak_season,porcini_score,chanterelle_score
41,21.340838,20.186633,17.908124,97.654124,34.602359,5.917170,10.168839,3,0,6,8.879973,147.441738,0.300227,1,0.639351,0.639351
84,15.262417,23.728837,17.390141,76.091168,28.590681,1.949583,19.477364,7,0,7,23.663798,91.888921,0.528805,1,1.000000,1.000000
85,15.731050,12.750235,16.449688,74.408951,15.095181,5.049115,36.621169,0,0,6,22.506349,14.131555,0.753803,1,0.622738,0.622738
135,15.665697,3.751010,25.475935,69.342689,11.380693,3.769862,8.186574,1,0,5,6.503117,74.722608,0.485821,1,0.651572,0.651572
141,13.308816,9.999612,17.438256,93.661329,21.438983,1.261153,28.933287,7,0,2,6.573957,27.649702,0.940401,1,0.550000,0.550000
163,22.680565,19.100690,24.353294,60.210499,42.298222,4.685570,29.784568,3,0,6,17.730673,107.615168,0.835246,1,0.882461,0.882461
166,15.585742,21.139086,19.585269,94.168850,30.425425,2.090974,32.592631,3,0,8,13.693474,35.129164,0.941140,1,0.956919,0.956919
177,17.762266,4.460555,19.786284,90.535249,79.060275,0.867066,13.665097,1,1,7,21.814026,146.397468,0.141549,1,0.706233,1.000000
206,8.348561,17.715794,11.440859,88.701142,59.772917,8.018402,20.684581,1,0,7,16.559145,41.185847,0.279442,1,0.568295,0.568295
262,9.622773,9.138325,25.528036,53.541733,45.260181,5.576887,12.529025,1,0,9,21.653895,146.740423,0.011199,1,0.521464,0.521464


In [7]:
import pandas as pd
import numpy as np

def generate_mushroom_dataset(n_samples=5000):
    """
    Generates a synthetic dataset for Porcini and Chanterelle growth probability
    based on the specific conditions provided.
    """
    np.random.seed(42)
    
    # 1. Generate Input Features (X)
    data = {
        'air_temp_day': np.random.uniform(5, 38, n_samples),
        'air_temp_night': np.random.uniform(2, 25, n_samples),
        'soil_temp': np.random.uniform(5, 28, n_samples),
        'air_humidity': np.random.uniform(20, 100, n_samples),
        'soil_moisture': np.random.uniform(10, 95, n_samples),
        'wind_speed': np.random.uniform(0, 10, n_samples),
        'rain_cumulative_7d': np.random.uniform(0, 80, n_samples),
        'rain_days_in_window': np.random.randint(0, 8, n_samples),
        'hours_above_28c': np.random.uniform(0, 24, n_samples),
        'consecutive_moist_days': np.random.randint(0, 15, n_samples),
        'is_peak_season': np.random.choice([0, 1], size=n_samples, p=[0.4, 0.6])
    }

    df = pd.DataFrame(data)

    def calculate_mushroom_logic(row, species='porcini'):
        score = 1.0
        
        # --- TEMPERATURE LOGIC ---
        # Optimal 15-22. Below 10-12 is min.
        if row['air_temp_day'] < 12:
            score *= np.interp(row['air_temp_day'], [5, 12], [0.1, 0.8])
        elif 15 <= row['air_temp_day'] <= 22:
            score *= 1.0
        elif row['air_temp_day'] > 22:
            # Decay towards suppression at 28
            score *= np.interp(row['air_temp_day'], [22, 28], [1.0, 0.3])
        
        # Night Temp (10-16 preferred)
        if not (10 <= row['air_temp_night'] <= 16):
            score *= 0.8

        # --- RAIN & MOISTURE LOGIC ---
        # Trigger: 15-30mm cumulative
        if 15 <= row['rain_cumulative_7d'] <= 30:
            score *= 1.2 
        elif row['rain_cumulative_7d'] > 60:
            score *= 0.4 # Soil saturation penalty
        else:
            score *= 0.5 # Sub-optimal rain

        # Light continuous rain vs storm (rain_days_in_window)
        # 3-5 days of rain out of 7 is better than 1 day of heavy rain
        if row['rain_days_in_window'] >= 3:
            score *= 1.1
        elif row['rain_days_in_window'] == 1:
            score *= 0.7

        # Soil Moisture (60-80% optimal)
        if 60 <= row['soil_moisture'] <= 80:
            score *= 1.0
        else:
            dist = min(abs(row['soil_moisture'] - 60), abs(row['soil_moisture'] - 80))
            score *= np.exp(-0.05 * dist)

        # Consecutive Moist Days (5-10 days requirement)
        if 5 <= row['consecutive_moist_days'] <= 10:
            score *= 1.0
        else:
            score *= 0.6

        # --- WIND & HUMIDITY ---
        if not (75 <= row['air_humidity'] <= 95):
            score *= 0.7
        if row['wind_speed'] > 3:
            score *= 0.5

        # --- SPECIES SPECIFIC KILL SWITCHES ---
        if species == 'porcini':
            if row['air_temp_day'] >= 32: 
                return 0.0 # Fruiting stops
        
        if species == 'chanterelle':
            if row['air_temp_day'] >= 28:
                score *= 0.2 # Heavily suppressed

        # --- SEASONALITY ---
        if row['is_peak_season'] == 0:
            score *= 0.05 # Growth very unlikely outside May-June/Sept-Oct
            
        return min(max(score, 0), 1.0)

    # Apply calculations to generate Targets (Y)
    df['porcini_score'] = df.apply(lambda r: calculate_mushroom_logic(r, 'porcini'), axis=1)
    df['chanterelle_score'] = df.apply(lambda r: calculate_mushroom_logic(r, 'chanterelle'), axis=1)

    return df

# Example Usage:
df = generate_mushroom_dataset(10000)
# print(df[['air_temp_day', 'rain_cumulative_7d', 'porcini_score', 'chanterelle_score']].head())

In [10]:
df[df["porcini_score"] > 0.5].head()

,air_temp_day,air_temp_night,soil_temp,air_humidity,soil_moisture,wind_speed,rain_cumulative_7d,rain_days_in_window,hours_above_28c,consecutive_moist_days,is_peak_season,porcini_score,chanterelle_score
275,21.550038,13.045444,11.294324,98.431296,62.188930,2.384293,17.558715,3,12.259565,4,1,0.554400,0.554400
302,22.840959,4.942849,20.061680,37.624875,62.185585,2.210528,20.685871,3,7.058361,5,1,0.666676,0.666676
343,19.317104,18.210304,6.448492,93.862147,71.563074,5.569570,17.252833,6,16.725043,8,1,0.528000,0.528000
375,15.526196,13.350508,17.806158,77.482906,63.008359,1.028835,23.696011,6,10.958522,0,1,0.792000,0.792000
496,24.260652,14.158486,14.313753,79.911218,80.943103,1.684893,17.457655,6,12.747277,4,1,0.556257,0.556257


In [15]:
df.sort_values(by=["hours_above_28c"], ascending=False)

,air_temp_day,air_temp_night,soil_temp,air_humidity,soil_moisture,wind_speed,rain_cumulative_7d,rain_days_in_window,hours_above_28c,consecutive_moist_days,is_peak_season,porcini_score,chanterelle_score
3533,14.879651,8.963374,26.093272,39.688739,21.245822,4.084104,67.775443,1,23.996813,3,0,0.000339,0.000339
7482,19.502357,18.100783,12.683804,73.456056,34.049491,6.369650,12.168463,7,23.996603,9,1,0.042074,0.042074
3156,35.897261,6.129151,16.669361,45.872636,38.478655,8.843463,8.820093,3,23.996224,4,0,0.000000,0.000095
55,35.421850,12.973985,27.213962,78.238106,55.629636,7.751865,19.181739,0,23.995631,2,0,0.000000,0.000868
640,5.662350,15.153003,12.686228,98.929029,18.238634,2.636676,73.487869,4,23.993055,6,1,0.006345,0.006345
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9172,6.052198,7.437291,17.765057,92.396615,78.996402,7.219123,25.543529,2,0.010497,0,0,0.002955,0.002955
2769,27.427471,10.260771,15.011261,83.105625,80.784902,0.580922,48.252991,0,0.009244,3,1,0.105804,0.105804
2985,36.912164,22.887037,23.317200,68.594685,12.003183,1.453901,5.110997,2,0.006494,5,1,0.000000,0.001524
1864,37.494101,20.153910,16.984833,42.576333,74.536050,0.527340,55.355324,6,0.006270,13,1,0.000000,0.011088


In [16]:
df.iloc[:, :-2]

,air_temp_day,air_temp_night,soil_temp,air_humidity,soil_moisture,wind_speed,rain_cumulative_7d,rain_days_in_window,hours_above_28c,consecutive_moist_days,is_peak_season
0,17.359824,10.593739,21.789961,71.051565,35.407524,8.472366,59.324416,5,0.045224,0,1
1,36.373572,9.656978,9.243776,56.743396,18.059511,4.945170,70.488150,2,23.039625,0,1
2,29.155800,6.051540,12.972713,97.159882,20.740534,1.954656,37.054390,4,12.847598,5,0
3,24.755730,15.967133,20.255455,37.518276,25.357046,7.366418,23.134299,1,16.809613,3,1
4,10.148615,12.962356,16.088055,67.028513,27.310533,4.186781,25.507726,5,21.902332,5,1
...,...,...,...,...,...,...,...,...,...,...,...
9995,33.302648,22.171888,26.902335,80.322713,62.907842,9.661407,47.551722,6,11.683360,3,0
9996,34.617792,3.076721,22.199957,81.162155,29.226454,3.732400,75.729262,0,20.083252,1,1
9997,36.241361,8.985065,5.447485,41.565555,73.664582,3.046749,73.673480,3,23.132739,7,1
9998,18.117104,12.196360,14.223111,54.745582,54.285928,4.073625,16.037149,3,0.105440,9,0


In [17]:
df.iloc[:, -2:]

,porcini_score,chanterelle_score
0,0.033772,0.033772
1,0.000000,0.000495
2,0.000649,0.000130
3,0.021173,0.021173
4,0.055409,0.055409
...,...,...
9995,0.000000,0.000396
9996,0.000000,0.001236
9997,0.000000,0.007392
9998,0.017359,0.017359


In [18]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import StandardScaler

def calculate_mushroom_logic(row, species='porcini'):
    """
    Core biological logic based on your specific environmental thresholds.
    Outputs a score between 0.0 and 1.0.
    """
    score = 1.0
    
    # 1. Temperature Logic (Optimal 15-22°C)
    if row['air_temp_day'] < 12:
        score *= np.interp(row['air_temp_day'], [5, 12], [0.1, 0.8])
    elif 15 <= row['air_temp_day'] <= 22:
        score *= 1.0
    elif row['air_temp_day'] > 22:
        score *= np.interp(row['air_temp_day'], [22, 28], [1.0, 0.3])
    
    # Night Temp Preferred (10-16°C)
    if not (10 <= row['air_temp_night'] <= 16):
        score *= 0.8

    # 2. Rain & Moisture Trigger (15-30mm cumulative)
    if 15 <= row['rain_cumulative_7d'] <= 30:
        score *= 1.2 
    elif row['rain_cumulative_7d'] > 60:
        score *= 0.3 # Saturation Penalty
    else:
        score *= 0.5 

    # Rain Quality (Light continuous > heavy storm)
    if row['rain_days_in_window'] >= 4:
        score *= 1.1
    elif row['rain_days_in_window'] <= 1:
        score *= 0.6

    # Soil Moisture (60-80% optimal)
    if not (60 <= row['soil_moisture'] <= 80):
        dist = min(abs(row['soil_moisture'] - 60), abs(row['soil_moisture'] - 80))
        score *= np.exp(-0.06 * dist)

    # 3. Species Specific "Kill Switches"
    if species == 'porcini':
        if row['air_temp_day'] >= 32: return 0.0 # Fruiting stops
    
    if species == 'chanterelle':
        if row['air_temp_day'] >= 28: score *= 0.1 # Growth suppressed

    # 4. Seasonality & Wind
    if row['is_peak_season'] == 0:
        score *= 0.05
    if row['wind_speed'] > 3:
        score *= 0.6
            
    return min(max(score, 0), 1.0)

def generate_perfect_dataset(n_samples=15000, save_path="data/mushrooms/scripts/"):
    np.random.seed(42)
    
    # Define ratios to fix the imbalance in the scatter plot
    n_perfect = int(n_samples * 0.45) # Force high-growth examples
    n_noise = n_samples - n_perfect

    # --- BUCKET 1: THE NOISE (Wide Random) ---
    df_noise = pd.DataFrame({
        'air_temp_day': np.random.uniform(5, 38, n_noise),
        'air_temp_night': np.random.uniform(2, 25, n_noise),
        'soil_temp': np.random.uniform(5, 28, n_noise),
        'air_humidity': np.random.uniform(20, 100, n_noise),
        'soil_moisture': np.random.uniform(10, 95, n_noise),
        'wind_speed': np.random.uniform(0, 8, n_noise),
        'rain_cumulative_7d': np.random.uniform(0, 80, n_noise),
        'rain_days_in_window': np.random.randint(0, 8, n_noise),
        'hours_above_28c': np.random.uniform(0, 12, n_noise),
        'consecutive_moist_days': np.random.randint(0, 15, n_noise),
        'is_peak_season': np.random.choice([0, 1], p=[0.5, 0.5], size=n_noise)
    })

    # --- BUCKET 2: THE SWEET SPOT (Forcing high-scores) ---
    df_perfect = pd.DataFrame({
        'air_temp_day': np.random.uniform(15.5, 21.5, n_perfect),
        'air_temp_night': np.random.uniform(11, 15, n_perfect),
        'soil_temp': np.random.uniform(14, 19, n_perfect),
        'air_humidity': np.random.uniform(78, 92, n_perfect),
        'soil_moisture': np.random.uniform(62, 78, n_perfect),
        'wind_speed': np.random.uniform(0.5, 2.8, n_perfect),
        'rain_cumulative_7d': np.random.uniform(18, 28, n_perfect),
        'rain_days_in_window': np.random.randint(4, 7, n_perfect),
        'hours_above_28c': np.zeros(n_perfect),
        'consecutive_moist_days': np.random.randint(6, 11, n_perfect),
        'is_peak_season': np.ones(n_perfect)
    })

    # Combine, Shuffle, and Score
    df = pd.concat([df_noise, df_perfect]).sample(frac=1).reset_index(drop=True)
    
    print("Calculating scores...")
    df['porcini_score'] = df.apply(lambda r: calculate_mushroom_logic(r, 'porcini'), axis=1)
    df['chanterelle_score'] = df.apply(lambda r: calculate_mushroom_logic(r, 'chanterelle'), axis=1)

    # --- SCALER PREPARATION ---
    feature_cols = [
        'air_temp_day', 'air_temp_night', 'soil_temp', 'air_humidity', 
        'soil_moisture', 'wind_speed', 'rain_cumulative_7d', 
        'rain_days_in_window', 'hours_above_28c', 'consecutive_moist_days', 
        'is_peak_season'
    ]

    # os.makedirs(save_path, exist_ok=True)
    # csv_file = os.path.join(save_path, "mushroom_growth_dataset_test.csv")
    # df.to_csv(csv_file, index=False)
    # print(f"Dataset saved to: {csv_file}")
    
    return df

if __name__ == "__main__":
    df_new = generate_perfect_dataset(n_samples=2000)

Calculating scores...


In [19]:
df_new.head()

,air_temp_day,air_temp_night,soil_temp,air_humidity,soil_moisture,wind_speed,rain_cumulative_7d,rain_days_in_window,hours_above_28c,consecutive_moist_days,is_peak_season,porcini_score,chanterelle_score
0,15.907557,11.930379,14.868197,91.742549,63.109060,1.039772,22.812435,5,0.000000,6,1.0,1.000000,1.000000
1,20.591526,12.155549,15.486719,79.147563,67.335312,0.556269,27.116830,5,0.000000,10,1.0,1.000000,1.000000
2,21.097775,14.696138,16.660599,85.638041,77.996882,1.249267,26.559759,5,0.000000,6,1.0,1.000000,1.000000
3,37.567269,8.601877,16.784367,75.571277,84.766127,7.795514,10.425439,6,5.939794,2,1.0,0.000000,0.005950
4,27.967149,23.541931,26.306800,26.961377,37.272636,0.453004,13.820585,7,6.504400,12,1.0,0.034187,0.034187


In [21]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/mushrooms/scripts/mushroom_growth_dataset.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual distributions
sns.histplot(data=df, x='porcini_score', kde=True, color='darkred', ax=axes[0], bins=5)
axes[0].set_title('Porcini Score Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

sns.histplot(data=df, x='chanterelle_score', kde=True, color='goldenrod', ax=axes[1], bins=5)
axes[1].set_title('Chanterelle Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/mushrooms/scripts/mushroom_growth_dataset.csv'